# nada

> Audio-JEPA audio embeddings on Modular MAX.


In [ ]:
[Audio-JEPA](https://arxiv.org/abs/2507.02915) on [Modular MAX](https://docs.modular.com/max/). This is the context encoder from `ltuncay/Audio-JEPA`: a ViT-Base that embeds a 10 s log-mel spectrogram as 128 tokens of width 768. The frontend is Kaldi fbank at 32 kHz. Predictor and target-encoder weights from the Lightning checkpoint are dropped; only `encoder.*` is used at inference.

The layout follows `solid_lens`. `config` is the architecture, `weights` binds a checkpoint, and `compile_model` traces the module for one batch size and one spectrogram size. The graph is a MAX module (`max.experimental.nn`): the same `forward` runs eagerly or compiled. Module and parameter names match the published encoder, so its state dict loads without renaming.


## Install

```sh
pixi install
pixi run -e reference convert
```

`convert` downloads `JEPA.ckpt` from Hugging Face and writes `models/audio_jepa.safetensors`. The Lightning file needs PyTorch to read, so conversion runs in the `reference` environment.


## Embed

```sh
pixi run embed clip.wav
```

The clip is 16-bit PCM WAV. 32 kHz is native; other rates are resampled. The same thing from Python:

```python
from nada.features import read_wav
from nada.model import build, compile_model, load_checkpoint, embed

clip = read_wav("clip.wav")
model = build()
load_checkpoint(model)
compiled = compile_model(model, batch=1)
print(embed(model, compiled, [clip])[0].shape)
```

`compiled` takes a float32 spectrogram of shape `(batch, 256, 128, 1)` and returns patch tokens `(batch, 128, 768)`. `embed` tiles a short clip across the 10 s canvas, mean-pools the rows that have sound, then subtracts `models/embed_center.npy`. The graph is compiled for one spectrogram size; the pretrained model is always 256 × 128.

Write the center file once after convert:

```sh
pixi run center
```


## Check

```sh
pixi run smoke
pixi run -e reference parity
```

`smoke` compiles a two-layer ViT and checks that the tokens are finite and the right shape. `parity` compares the numpy Kaldi frontend with `torchaudio`, and the MAX encoder with a PyTorch ViT of the same weights. The real checkpoint is compared too once it is converted.
